In [0]:
import requests, zipfile, io

HEADERS = {"User-Agent": "Your Name your.email@example.com"}
BASE = "https://www.sec.gov/files/dera/data/financial-statement-notes-data-sets"

r = requests.get(f"{BASE}/2009q2_notes.zip", headers=HEADERS, timeout=60)
print("status:", r.status_code, "| bytes:", len(r.content))

z = zipfile.ZipFile(io.BytesIO(r.content))
print(z.namelist())

In [0]:
r = requests.get(f"{BASE}/2024q1_notes.zip", headers=HEADERS, stream=True, timeout=60)
print("status:", r.status_code, "| size MB:", int(r.headers.get("Content-Length", 0)) / 1e6)
r.close()

In [0]:
import os
out_dir = "/Workspace/Users/poppet982398@gmail.com/Drafts/sec_test"
os.makedirs(out_dir, exist_ok=True)
z.extract("sub.tsv", out_dir)      # use the exact name Cell 1 printed
df = (spark.read.option("sep", "\t").option("header", True)
      .csv(f"{out_dir}/sub.tsv"))
print(df.count())

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
out = df.withColumn("period", F.lit("2009q2"))

out.write.format("delta").mode("overwrite").partitionBy("period") \
   .saveAsTable("bronze.test_sub")

# the idempotent pattern Phase 1 relies on: overwrite one period only
out.write.format("delta").mode("overwrite") \
   .option("replaceWhere", "period = '2009q2'").saveAsTable("bronze.test_sub")

print(spark.table("bronze.test_sub").count())   # should still be 26

In [0]:
print(spark.sql("SELECT current_catalog(), current_schema()").collect())

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.sec_raw")
display(spark.sql("SHOW VOLUMES IN workspace.default"))

In [0]:
import os, shutil, time
src = "/tmp/sec_big"
dst = "/Volumes/workspace/default/sec_raw/2024q1"
os.makedirs(dst, exist_ok=True)

t = time.time()
for name in ["sub.tsv", "dim.tsv", "num.tsv"]:
    shutil.copy(f"{src}/{name}", f"{dst}/{name}")
    print(name, round(os.path.getsize(f"{dst}/{name}") / 1e6), "MB")
print("seconds:", round(time.time() - t))

In [0]:
dst = "/Volumes/workspace/default/sec_raw/2024q1"

def read_tsv(name):
    return (spark.read
            .option("sep", "\t")          # columns are separated by tabs
            .option("header", True)       # first line is the column names
            .option("quote", "")          # don't treat " as special
            .option("inferSchema", False) # keep everything as text
            .csv(f"{dst}/{name}"))

for name in ["sub.tsv", "dim.tsv", "num.tsv"]:
    print(name, read_tsv(name).count())

In [0]:
lines = spark.read.text(f"{dst}/dim.tsv").count() - 1   # minus the header line
rows = read_tsv("dim.tsv").count()
print("lines in file:", lines, "| rows Spark parsed:", rows)

In [0]:
from itertools import islice
import pyspark.sql.functions as F

# Line 185897 of the file (Python counts from 0, so we ask for index 185896)
with open(f"{dst}/dim.tsv", encoding="utf-8") as f:
    raw = next(islice(f, 185896, 185897))

print("raw line:", repr(raw))
print("tabs in this line:", raw.count("\t"), "(a normal row has 2)")

# What did Spark make of the same row?
key = raw.split("\t")[0]
read_tsv("dim.tsv").filter(F.col("dimhash") == key).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/default/sec_raw"
TABLES = ["sub", "num", "dim"]

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

def load_bronze(period, table):
    path = f"{VOLUME}/{period}/{table}.tsv"
    df = (spark.read
          .option("sep", "\t").option("header", True)
          .option("quote", "").option("inferSchema", False)
          .csv(path))
    df = (df.withColumn("period", F.lit(period))
            .withColumn("_source_file", F.lit(path))
            .withColumn("_ingested_at", F.current_timestamp()))
    target = f"bronze.{table}"
    if spark.catalog.tableExists(target):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"period = '{period}'")
           .saveAsTable(target))
    else:
        (df.write.format("delta").mode("overwrite")
           .partitionBy("period").saveAsTable(target))
    n = spark.table(target).filter(F.col("period") == period).count()
    print(f"{period} | bronze.{table} | rows for this period: {n}")

# ---- This is the line that actually does the work ----
for p in ["2024q1", "2023q4"]:
    for t in TABLES:
        load_bronze(p, t)

In [0]:
display(spark.sql("SHOW TABLES IN workspace.bronze"))

In [0]:
def total(t):
    return spark.table(f"bronze.{t}").count()

before = {t: total(t) for t in TABLES}

for t in TABLES:
    load_bronze("2024q1", t)          # load the same quarter AGAIN

after = {t: total(t) for t in TABLES}

print("before:", before)
print("after: ", after)
print("idempotent?", before == after)

In [0]:
import requests, zipfile, os, shutil, time
from pyspark.sql import functions as F

VOLUME  = "/Volumes/workspace/default/sec_raw"
BASE    = "https://www.sec.gov/files/dera/data/financial-statement-notes-data-sets"
HEADERS = {"User-Agent": "poppet982398@gmail.com"}   # <- CHANGE THIS
TABLES  = ["sub", "num", "dim"]
PERIODS = ["2023q1", "2023q2", "2023q3", "2024q2", "2024q3", "2024q4"]

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

def download_period(period):
    d = f"{VOLUME}/{period}"
    os.makedirs(d, exist_ok=True)
    if all(os.path.exists(f"{d}/{t}.tsv") for t in TABLES):
        return
    zpath = f"/tmp/{period}_notes.zip"
    with requests.get(f"{BASE}/{period}_notes.zip", headers=HEADERS,
                      stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(zpath, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    with zipfile.ZipFile(zpath) as z:
        for t in TABLES:
            with z.open(f"{t}.tsv") as src, open(f"{d}/{t}.tsv", "wb") as out:
                shutil.copyfileobj(src, out)
    os.remove(zpath)

def load_bronze(period, table):
    path = f"{VOLUME}/{period}/{table}.tsv"
    df = (spark.read.option("sep", "\t").option("header", True)
          .option("quote", "").option("inferSchema", False).csv(path))
    df = (df.withColumn("period", F.lit(period))
            .withColumn("_source_file", F.lit(path))
            .withColumn("_ingested_at", F.current_timestamp()))
    target = f"bronze.{table}"
    if spark.catalog.tableExists(target):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"period = '{period}'").saveAsTable(target))
    else:
        (df.write.format("delta").mode("overwrite")
           .partitionBy("period").saveAsTable(target))
    return spark.table(target).filter(F.col("period") == period).count()

def log(period, table, status, rows, secs, msg=""):
    (spark.createDataFrame(
        [(period, table, status, rows, round(secs, 1), msg[:500])],
        "period string, table_name string, status string, row_count long, seconds double, message string")
     .withColumn("logged_at", F.current_timestamp())
     .write.format("delta").mode("append").saveAsTable("bronze.ingestion_log"))

for p in PERIODS:
    t0 = time.time()
    try:
        download_period(p)
    except Exception as e:
        log(p, "ALL", "FAILED", None, time.time() - t0, f"download: {e}")
        print(p, "download FAILED:", e)
        continue
    for t in TABLES:
        t1 = time.time()
        try:
            n = load_bronze(p, t)
            log(p, t, "SUCCESS", n, time.time() - t1)
            print(f"{p} | {t} | {n} rows")
        except Exception as e:
            log(p, t, "FAILED", None, time.time() - t1, str(e))
            print(f"{p} | {t} FAILED:", e)

display(spark.sql("SELECT * FROM bronze.ingestion_log ORDER BY logged_at"))

In [0]:
display(spark.sql("""
  SELECT period, COUNT(*) AS filings, COUNT(DISTINCT adsh) AS distinct_filings
  FROM bronze.sub GROUP BY period ORDER BY period
"""))

In [0]:
display(spark.sql("""
  SELECT period, name, form, fy, fp, period AS load_period
  FROM bronze.sub
  WHERE CAST(cik AS BIGINT) IN (320193, 1018724, 789019)
    AND form IN ('10-K', '10-Q')
  ORDER BY period, name
"""))

In [0]:
from pyspark.sql import functions as F
import time

VOLUME  = "/Volumes/workspace/default/sec_raw"
TABLES  = ["sub", "num", "dim"]
PERIODS = ["2023q1", "2023q2", "2023q3", "2023q4",
           "2024q1", "2024q2", "2024q3", "2024q4"]

# Start clean: drop the three Bronze tables (and the old test table).
# The raw files in the Volume and the ingestion log are untouched.
for t in TABLES + ["test_sub"]:
    spark.sql(f"DROP TABLE IF EXISTS bronze.{t}")

def load_bronze(period, table):
    path = f"{VOLUME}/{period}/{table}.tsv"
    df = (spark.read.option("sep", "\t").option("header", True)
          .option("quote", "").option("inferSchema", False).csv(path))
    df = (df.withColumn("load_period", F.lit(period))      # renamed from "period"
            .withColumn("_source_file", F.lit(path))
            .withColumn("_ingested_at", F.current_timestamp()))
    target = f"bronze.{table}"
    if spark.catalog.tableExists(target):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"load_period = '{period}'").saveAsTable(target))
    else:
        (df.write.format("delta").mode("overwrite")
           .partitionBy("load_period").saveAsTable(target))
    return spark.table(target).filter(F.col("load_period") == period).count()

def log(period, table, status, rows, secs, msg=""):
    (spark.createDataFrame(
        [(period, table, status, rows, round(secs, 1), msg[:500])],
        "period string, table_name string, status string, row_count long, seconds double, message string")
     .withColumn("logged_at", F.current_timestamp())
     .write.format("delta").mode("append").saveAsTable("bronze.ingestion_log"))

for p in PERIODS:
    for t in TABLES:
        t1 = time.time()
        try:
            n = load_bronze(p, t)
            log(p, t, "SUCCESS", n, time.time() - t1)
            print(f"{p} | {t} | {n} rows")
        except Exception as e:
            log(p, t, "FAILED", None, time.time() - t1, str(e))
            print(f"{p} | {t} FAILED:", e)

# Proof the original column is back: balance date (period) next to our load_period
display(spark.sql("""
  SELECT load_period, period AS balance_date, name, form
  FROM bronze.sub
  WHERE CAST(cik AS BIGINT) = 1018724 AND form IN ('10-K', '10-Q')
  ORDER BY load_period
"""))